In [14]:
# Setup et connexion MinIO
from pyspark.sql import SparkSession

In [15]:
# Spark avec connecteur MinIO (S3)
# spark = SparkSession.builder \
#     .master("local[*]") \
#     .appName("HospitalOptimizer-ETL-Bronze") \
#     .config("spark.driver.memory", "4g") \
#     .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
#     .config("spark.hadoop.fs.s3a.endpoint", "http://hospital-minio:9000") \
#     .config("spark.hadoop.fs.s3a.access.key", "hospital") \
#     .config("spark.hadoop.fs.s3a.secret.key", "hospital123") \
#     .config("spark.hadoop.fs.s3a.path.style.access", "true") \
#     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
#     .getOrCreate()

# spark.sparkContext.setLogLevel("ERROR")
# print(f"✅ Spark {spark.version} a démarré")


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-ETL") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} démarré")

✅ Spark 3.5.0 démarré


In [16]:
import pandas as pd

# Lire le fichier source
df_raw = spark.read.csv(
    '/home/jovyan/data/raw/mimic3d.csv',
    header=True,
    inferSchema=True
)

print(f"✅ Extraction réussie")
print(f"   Lignes   : {df_raw.count():,}")
print(f"   Colonnes : {len(df_raw.columns)}")

✅ Extraction réussie
   Lignes   : 58,976
   Colonnes : 28


In [10]:
# stop 
spark.stop()

In [18]:
# 4 — Sauvegarde Bronze en local
# df_raw.write \
#     .mode("overwrite") \
#     .parquet("s3a://bronze/mimic3d_raw.parquet")

# print("✅ Bronze chargé dans MinIO")
# print("   Chemin : s3a://bronze/mimic3d_raw.parquet")
# print("   Format : Parquet")

df_raw.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/bronze/mimic3d_raw.parquet")

print("✅ Bronze sauvegardé")
print("   Chemin : data/bronze/mimic3d_raw.parquet")

✅ Bronze sauvegardé
   Chemin : data/bronze/mimic3d_raw.parquet


In [19]:
# ── Couche Silver : nettoyage des données ─────────────────────────

# 1. Lire depuis Bronze
df_bronze = spark.read.parquet("/home/jovyan/data/bronze/mimic3d_raw.parquet")

# 2. Imputer les valeurs manquantes
df_silver = df_bronze \
    .fillna("UNKNOWN", subset=["marital_status", "religion", "AdmitDiagnosis"]) \

# 3. Supprimer les colonnes inutiles (data leakage + redondance)
cols_to_drop = ["LOSgroupNum", "TotalNumInteract"]
df_silver = df_silver.drop(*cols_to_drop)

# 4. Filtrer les outliers extrêmes sur LOSdays
df_silver = df_silver.filter(df_silver.LOSdays <= 200)

# 5. Vérifications
print(f"✅ Silver prêt")
print(f"   Lignes   : {df_silver.count():,}")
print(f"   Colonnes : {len(df_silver.columns)}")

✅ Silver prêt
   Lignes   : 58,973
   Colonnes : 26


In [20]:
# Sauvegarder Silver
df_silver.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/silver/mimic3d_clean.parquet")

print("✅ Silver sauvegardé")
print("   Chemin : data/silver/mimic3d_clean.parquet")

✅ Silver sauvegardé
   Chemin : data/silver/mimic3d_clean.parquet


In [22]:
# features métier
from pyspark.sql import functions as F

# 1. Lire depuis Silver
df_gold = spark.read.parquet("/home/jovyan/data/silver/mimic3d_clean.parquet")

# 2. Feature : score de complexité du patient
df_gold = df_gold.withColumn(
    "complexity_score",
    F.round(
        (df_gold.NumDiagnosis + df_gold.NumProcs + df_gold.NumRx) / 3, 2
    )
)

# 3. Feature : charge de travail infirmier
df_gold = df_gold.withColumn(
    "nursing_workload",
    F.round(
        (df_gold.NumChartEvents + df_gold.NumOutput + df_gold.NumInput) / 3, 2
    )
)

# 4. Feature : patient à risque (1 si LOSdays > médiane)
df_gold = df_gold.withColumn(
    "high_risk",
    F.when(df_gold.LOSdays > 6.5, 1).otherwise(0)
)

# Vérification
print(f"✅ Gold prêt")
print(f"   Lignes   : {df_gold.count():,}")
print(f"   Colonnes : {len(df_gold.columns)}")
print(f"\nNouvelles features :")
df_gold.select("LOSdays", "complexity_score", "nursing_workload", "high_risk").show(5)

✅ Gold prêt
   Lignes   : 58,973
   Colonnes : 29

Nouvelles features :
+-------+----------------+----------------+---------+
|LOSdays|complexity_score|nursing_workload|high_risk|
+-------+----------------+----------------+---------+
|   6.17|            5.83|           143.0|        0|
|   4.04|            3.47|          130.69|        0|
|  12.04|            2.38|          100.61|        1|
|   7.29|            4.16|          185.14|        1|
|   4.88|            9.84|           197.2|        0|
+-------+----------------+----------------+---------+
only showing top 5 rows



In [23]:
# Sauvegarder Gold
df_gold.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/gold/mimic3d_features.parquet")

print("✅ Gold sauvegardé")
print("   Chemin : data/gold/mimic3d_features.parquet")

✅ Gold sauvegardé
   Chemin : data/gold/mimic3d_features.parquet


In [24]:
# Gestion des erreurs dans le pipeline ETL
# Un pipeline robuste doit gérer les erreurs proprement et ne pas crasher si quelque chose se passe mal.

import logging

# Configuration du logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("HospitalETL")

def extract(path):
    """Lit le fichier source avec gestion d'erreurs."""
    try:
        df = spark.read.csv(path, header=True, inferSchema=True)
        logger.info(f"✅ Extract OK — {df.count():,} lignes")
        return df
    except Exception as e:
        logger.error(f"❌ Extract échoué : {e}")
        return None

def transform_silver(df):
    """Nettoie les données brutes."""
    try:
        df = df.fillna("UNKNOWN", subset=["marital_status", "religion", "AdmitDiagnosis"])
        df = df.drop("LOSgroupNum", "TotalNumInteract")
        df = df.filter(df.LOSdays <= 200)
        logger.info(f"✅ Transform Silver OK — {df.count():,} lignes")
        return df
    except Exception as e:
        logger.error(f"❌ Transform Silver échoué : {e}")
        return None

def transform_gold(df):
    """Crée les features métier."""
    try:
        df = df.withColumn("complexity_score",
            F.round((df.NumDiagnosis + df.NumProcs + df.NumRx) / 3, 2))
        df = df.withColumn("nursing_workload",
            F.round((df.NumChartEvents + df.NumOutput + df.NumInput) / 3, 2))
        df = df.withColumn("high_risk",
            F.when(df.LOSdays > 6.5, 1).otherwise(0))
        logger.info(f"✅ Transform Gold OK — {df.count():,} lignes")
        return df
    except Exception as e:
        logger.error(f"❌ Transform Gold échoué : {e}")
        return None

def load(df, path):
    """Sauvegarde en Parquet."""
    try:
        df.write.mode("overwrite").parquet(path)
        logger.info(f"✅ Load OK — {path}")
    except Exception as e:
        logger.error(f"❌ Load échoué : {e}")

# ── Exécution du pipeline complet ────────────────────────────────
logger.info("=== Démarrage pipeline ETL ===")

df_raw    = extract("/home/jovyan/data/raw/mimic3d.csv")
df_silver = transform_silver(df_raw)
df_gold   = transform_gold(df_silver)

load(df_silver, "/home/jovyan/data/silver/mimic3d_clean.parquet")
load(df_gold,   "/home/jovyan/data/gold/mimic3d_features.parquet")

logger.info("=== Pipeline ETL terminé ===")

INFO:HospitalETL:=== Démarrage pipeline ETL ===
INFO:HospitalETL:✅ Extract OK — 58,976 lignes
INFO:HospitalETL:✅ Transform Silver OK — 58,973 lignes
INFO:HospitalETL:✅ Transform Gold OK — 58,973 lignes
INFO:HospitalETL:✅ Load OK — /home/jovyan/data/silver/mimic3d_clean.parquet
INFO:HospitalETL:✅ Load OK — /home/jovyan/data/gold/mimic3d_features.parquet
INFO:HospitalETL:=== Pipeline ETL terminé ===
